### 単一元素結晶の分類問題


1. hcp (red)
2. bcc (blue)
3. fcc (green)

の３つの結晶構造を分類する問題である。


|1|2|3|4|5|6|7|8|9|10|11|12|13|14|15|16|17|18|
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
|<font color="red">H</font>| _ | _ |_ |_ |_ |_ |_ |_ |_ |_ |_ |_ |_ |_ |_ |_ |<font color="red">He</font>|
|<font color="blue">Li</font>|<font color="red">Be</font>|_|_|_|_|_|_|_|_|_|_|B|C|N|O|F|<font color="green">Ne</font>|
|<font color="blue">Na</font>|<font color="red">Mg</font>|_|_|_|_|_|_|_|_|_|_|<font color="green">Al</font>|Si|P|S|Cl|<font color="green">Ar</font>|
|<font color="blue">K</font>|<font color="green">Ca</font>|<font color="red">Sc</font>|<font color="red">Ti</font>|<font color="blue">V</font>|<font color="blue">Cr</font>|Mn|<font color="blue">Fe</font>|<font color="red">Co</font>|<font color="green">Ni</font>|<font color="green">Cu</font>|<font color="red">Zn</font>|Ga|Ge|As|Se|Br|<font color="green">Kr</font>|
|<font color="blue">Rb</font>|<font color="green">Sr</font>|<font color="red">Y</font>|<font color="red">Zr</font>|<font color="blue">Nb</font>|<font color="blue">Mo</font>|<font color="red">Tc</font>|<font color="red">Ru</font>|<font color="green">Rh</font>|<font color="green">Pd</font>|<font color="green">Ag</font>|<font color="red">Cd</font>|In|Sn|Sb|Te|I|<font color="green">Xe</font>|
|<font color="blue">Cs</font>|<font color="blue">Ba</font>|_|<font color="red">Hf</font>|<font color="blue">Ta</font>|<font color="blue">W</font>|<font color="red">Re</font>|<font color="red">Os</font>|<font color="green">Ir</font>|<font color="green">Pt</font>|<font color="green">Au</font>|Hg|<font color="red">Tl</font>|<font color="green">Pb</font>|Bi|Po|At|Rn|
|Fr|Ra|_|_|_|_|_|_|_|_|_|_|_|_|_|_|_|_|
|_|_|La|<font color="green">Ce</font>|Pr|Nd|Pm|Sm|<font color="blue">Eu</font>|<font color="red">Gd</font>|<font color="red">Tb</font>|<font color="red">Dy</font>|<font color="red">Ho</font>|<font color="red">Er</font>|<font color="red">Tm</font>|<font color="green">Yb</font>|<font color="red">Lu</font>|_|
|_|_|<font color="green">Ac</font>|<font color="green">Th</font>|Pa|U|Np|Pu|Am|Cm|Bk|Cf|Es|Fm|Md|No|Lr|_|


**データ取得からデータ解析**

In [ ]:
from sklearn.metrics import confusion_matrix
from sklearn.linear_model import LogisticRegressionCV, LogisticRegression
from sklearn.multiclass import OneVsRestClassifier

from sklearn.metrics import classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
import warnings
import numpy as np
import pandas as pd
import matplotlib.pylab as plt
%matplotlib inline
pd.set_option('display.max_rows', 100)
warnings.filterwarnings('ignore')


In [ ]:
# データ取得
g_dfraw = pd.read_csv("../data/mono_structure.csv")
g_descriptor_names = ['min_oxidation_state', 'max_oxidation_state', 'row',
                     'group', 's', 'p', 'd', 'f', 'atomic_radius_calculated', 'X', 'IP',
                     'EA']
g_target_name = 'crystal_structure'

def convert_crystaltype(dfraw, target_name, 
                        target_str = {0: "misc", 1:"hcp", 2:"bcc", 3:"fcc"}):
    """       0: misc (black)
       1: hcp (red)
       2: bcc (blue)
       3: fcc (green)
       の変換を行う。

    Args:
        dfraw (pd.DataFrame): データ.
        target_name ([str]): 目的変数名
        target_str (dict, optional): 変換辞書. Defaults to {0: "misc", 1:"hcp", 2:"bcc", 3:"fcc"}.

    Returns:
        pd.DataFrame: 目的変数を変換されたデータ
    """
    targets = dfraw[target_name].values
    targetlist = []
    for target in targets:
        targetlist.append(target_str[target])
    targetlist
    dfraw[target_name] = targetlist
    return dfraw

g_dfraw =  convert_crystaltype(g_dfraw, g_target_name)
g_dfraw

簡単のため結晶構造が0: miscとなるデータを除く。

In [ ]:
# df = dfraw[dfraw["crystal_structure"] != 0].reset_index(drop=True)
g_df = g_dfraw[g_dfraw["crystal_structure"] != "misc"].reset_index(drop=True)

In [ ]:
def predict_and_score(df, descriptor_names, target_name, multi_class="ovr"):
    """fitとpredictを行う。

    Args:
        df (pd.DataFrame): データ.
        descriptor_names ([str]]): a list of explanaroty variables names.
        target_name (str): target variable.

    Returns:
        np.ndarray: explanaroty variables.
        np.ndarray: observed target variables.
        np.ndarray: predicted target variables.
        np.ndarray: predicted probability of target variables.
        LogisticRegressionCV: LogisticRegressionCV instance
    """
    Xraw = df.loc[:, descriptor_names].values
    y = df.loc[:, target_name].values

    # データプリプロセス
    scaler = StandardScaler()
    X = scaler.fit(Xraw)
    X = scaler.transform(Xraw)

    # データ解析
    kf = KFold(5, shuffle=True)
    # cls = LogisticRegressionCV(cv=kf, refit=True, multi_class='ovr')
    if multi_class=="ovr":
        print("multi_class=ovr")
        cls = OneVsRestClassifier(
            LogisticRegressionCV(cv=kf, refit=True)
        )    
    else:
        print("multi_class=multinomial")
        cls = LogisticRegressionCV(cv=kf, refit=True)

    cls.fit(X, y)
    score = cls.score(X, y)
    print("score", score)
    yp = cls.predict(X)
    yproba = cls.predict_proba(X)
    print(classification_report(y, yp, digits=3))
    with open("image_executed/mono_structure_cls_report.txt", "w") as f:
        f.write(classification_report(y, yp, digits=3))
    index_names = []
    column_names = []
    for name in cls.classes_:
        index_names.append(f"actual({name})")
        column_names.append(f"predict({name})")
        
    cmdf = pd.DataFrame(confusion_matrix(y, yp), index=index_names,
                        columns=column_names)
    display(cmdf)
    return X, y, yp, yproba, cls 

g_X, g_y,g_yp,g_yproba,g_cls = predict_and_score(g_df, g_descriptor_names, g_target_name)

#### 可視化

規格化された説明変数の値の範囲と目的変数の頻度を表示します。

In [ ]:
def plot_X(X):
    """説明変数の図示。

    Args:
        X (np.ndarray): 説明変数
    """    
    fig, ax = plt.subplots()
    ax.plot(X)
    ax.set_xlabel("index")
    ax.set_ylabel("X")
    
plot_X(g_X)

def hist_y(y):
    """目的変数のhistogram図示。

    Args:
        y (np.ndarray): 目的変数
    """    
    plt.figure()
    plt.hist(y)
    plt.xlabel("y")

hist_y(g_y)

OvR法を用いた場合には最適なハイパーパラメタが３つあります。

In [ ]:
try:
    for label, estimators in zip(g_cls.classes_, g_cls.estimators_):
        print(f"{label}, C={estimators.C_}")
except AttributeError:
    print("Not C_ parameter")

In [ ]:
def show_CV_score_multi(cls, uniquey, save_fig: bool=False):
    plt.figure()
    plt.xlabel("log10(C)")
    plt.ylabel("score")

    # Case 1: OneVsRestClassifier(LogisticRegressionCV)
    if hasattr(cls, "estimators_"):
        for i, ytarget in enumerate(uniquey):
            est = cls.estimators_[i]
            key = 1 if 1 in est.scores_ else list(est.scores_.keys())[0]

            score_mean = np.mean(est.scores_[key], axis=0)
            score_std = np.std(est.scores_[key], axis=0)

            plt.errorbar(
                np.log10(est.Cs_) + 0.1 * i,
                score_mean,
                yerr=score_std,
                capsize=5,
                label=str(ytarget),
            )

            ic = np.argmax(score_mean)
            print("y=", ytarget, "Copt=", est.Cs_[ic], "selected C=", est.C_[0])

    # Case 2: LogisticRegressionCV
    else:
        for i, ytarget in enumerate(uniquey):
            score_mean = np.mean(cls.scores_[ytarget], axis=0)
            score_std = np.std(cls.scores_[ytarget], axis=0)

            plt.errorbar(
                np.log10(cls.Cs_) + 0.1 * i,
                score_mean,
                yerr=score_std,
                capsize=5,
                label=str(ytarget),
            )

            ic = np.argmax(score_mean)
            print("y=", ytarget, "Copt=", cls.Cs_[ic])

    plt.legend()

    if save_fig:
        plt.savefig("image_executed/mono_structure_hyperparameter_vs_score.png")

    plt.show()
    
show_CV_score_multi(g_cls,np.unique(g_y))

### 付録

boxplotでも見ておきます。

In [ ]:
def plot_CV_scores_as_boxplot(cls, y):
    """
    LogisticRegressionCV または
    OneVsRestClassifier(LogisticRegressionCV)
    の CV score を boxplot 表示する。
    """

    # Case 1: OneVsRestClassifier(LogisticRegressionCV)
    if hasattr(cls, "estimators_"):
        for i, ytarget in enumerate(cls.classes_):
            est = cls.estimators_[i]

            labels = [f"{x:.2f}" for x in np.log10(est.Cs_)]

            key = 1 if 1 in est.scores_ else list(est.scores_.keys())[0]

            fig, ax = plt.subplots()
            ax.set_title(f"target={ytarget}")

            df_score = pd.DataFrame(est.scores_[key], columns=labels)
            df_score.boxplot(rot=90, ax=ax)

            ax.set_xlabel("log10(C)")
            ax.set_ylabel("score")

    # Case 2: LogisticRegressionCV
    else:
        labels = [f"{x:.2f}" for x in np.log10(cls.Cs_)]

        for ytarget in np.unique(y):
            fig, ax = plt.subplots()
            ax.set_title(f"target={ytarget}")

            df_score = pd.DataFrame(cls.scores_[ytarget], columns=labels)
            df_score.boxplot(rot=90, ax=ax)

            ax.set_xlabel("log10(C)")
            ax.set_ylabel("score")

plot_CV_scores_as_boxplot(g_cls, g_y)

### 実験値と予測値の比較

線が重なっていないところが予測に失敗している物質です。
その後に具体的な元素名と詳細を表示しています。

actualでなくtrueの場合もよくある。

In [ ]:
%matplotlib inline


def plot_y(y, y_predict, proba, symbols, labels):
    """plot y vs y_predict.

    Args:
        y (np.array): target values.
        y_predict (np.array): predicted target values.
        proba (np.array): probability.
        symbols ([str]): material name
        labels ([str]): target labels.
    """
    plt.plot(y, "b-", label="y")
    plt.plot(y_predict, "r-", label="predict_y")
    plt.legend()
    plt.show()
    failedlist = []
    for i, (p1, p2, pro, s) in enumerate(zip(y, y_predict, proba, symbols)):
        if p1 != p2:
            failed = [i, s, p1, p2]
            failed.extend(pro)
            failedlist.append(failed)
            
    columns = ["index","element","actual","pred"]
    for i in labels:
        columns.append("P({})".format(i))
    
    return pd.DataFrame(failedlist, columns=columns)
    
g_df_failed = plot_y(g_y, g_yp, g_yproba, g_df["symbol"], g_cls.classes_)
print("failed at ")
g_df_failed

In [ ]:
# 確率の表示

def plot_proba(y, yp, proba, labels):
    """plot y vs yp and probability

    Args:
        y (np.array): target values
        yp (np.array): predicted target values
        proba (np.array): probability
    """
    fig, axes = plt.subplots(2,1,figsize=(12, 8))
    ax = axes[0]
    ax.plot(y, "o-", label="$y^{obs}$")
    ax.plot(yp, "o-", label="$y^{pred}$")
    ax.legend()
    
    ax = axes[1]
    n = proba.shape[1]
    for i in range(n):
        ax.plot(proba[:, i], "o-", label=labels[i])

    ax.legend()
    fig.show()


plot_proba(g_y, g_yp, g_yproba, g_cls.classes_)

In [ ]:
# 失敗したデータの内訳を示します。
def show_failed_data(cls, df_failed):
    """show df_failed

    Args:
        df_failed (pd.DataFrame): 説明変数

    Returns:
        pd.DataFrame: データ
    """
    classes = cls.classes_.tolist()
    occur = np.zeros( (len(classes), len(classes)) )
    for act,pred in zip(df_failed["actual"],df_failed["pred"]):
        i1 = classes.index(act)
        i2 = classes.index(pred)
        occur[i1,i2] +=1
    index = []
    columns = []
    for s in classes:
        index.append("actual({})".format(s))
        columns.append("pred({})".format(s))

    # 混同行列ではありません。
    df = pd.DataFrame(occur, index=index, columns=columns).astype(int)
    return df

show_failed_data(g_cls, g_df_failed)

（上の行列は混同行列ではありません。）

この表によると、
1. hcpと誤って予測する場合が多いことが分かります。
2. fccとhcpとを混同する場合が多いことが分かります。

説明変数に構造は入っていませんが、
2.についてはそもそもfccとhcpは構造が似ているせいかもしれません。
fccの(111)面がhcpの(0001)面(c軸方向）に対応しており、
それぞれの面（軸方向）にfccはABCABCと重なり、一方、hcpはABABと重なっているという意味で似ています。（全エネルギーの差が近いということにはなりません。）

ref. 
- https://www.researchgate.net/publication/229086534_Ab_Initio_Discovery_of_Novel_Crystal_Structure_Stability_in_Barium_andSodium-Calcium_Compounds_under_Pressure_using_DFT/figures?lo=1
- https://en.wikipedia.org/wiki/Stacking_fault

そもそも分類性能値が良くないので、説明変数の作り方に根本的な問題があるのかもしれません。

**コメント**

LogisticRegressionCVで最後につくれらた回帰モデルはクラスごとに別のCを使用しており、
単一ハイパーパラメタを与えて作成するLogisticRegressionよりも最適化が進んでいるため回帰性能が高くなっています。

## scikit-learn v1.8.0対応版
LogisticRegressionはmultinomialのみ。
OvRを利用するにはOneVsRestClassifierを用いた。